In [64]:
from collections import deque
import random


class Node:
    '''
    Represents a state in the search tree.

    state: tuple representing board configuration
    parent: pointer to parent node (used to reconstruct solution path)
    move: move that generated the state
    depth: depth of node in search tree
    '''
    def __init__(self, state, parent =None, move =None, depth =0):
        self.state = state
        self.parent = parent
        self.move = move
        self.depth = depth

In [65]:
def random_board_generator():
    '''
    Generate starting state and checks if the state is solvable.

    A configuration of the 8-puzzle is solvable if the number of inversions
    is even.

    This function repeatedly shuffles the board until a solvable configuration
    is produced, then returns it as an immutable tuple.
    '''

    # Starts with a list of the goal state (0 represents the blank)
    board = [1, 2, 3, 4, 0, 5, 6, 7, 8]
    solvable = False

    # Continue generating until a solvable board is found
    while not solvable:
        no_zero = []                
        inversion_count = 0        
        random.shuffle(board)       
        
        # Count number of inversions in the shuffled board
        for i in board:             
            no_zero.append(i)

        # Create a copy of the board excluding 0
        no_zero.remove(0)           

        # Counts pairs (i,j) where i appears before j but i > j
        for a in range(len(no_zero)):
            for b in range(a + 1, len(no_zero)):        
                if no_zero[a] > no_zero[b]:
                    inversion_count += 1

        # Check if inversions are even
        if inversion_count % 2 == 0:                    
            solvable = True
            return tuple(board)  # Return as tuple for hashability in sets                 

def print_state(board):
    '''
    Prints the 8-puzzle board in a 3x3 grid format.

    The board is stored internally as a flat tuple of length 9.

    The blank tile (0) is diplayed as "_" for clarity.
    '''

    #iterate over the board in groups of 3 to form rows
    for i in range(0,9,3):
        chunk = board[i:i+3]

        row = []

        # Replace 0 with "_" fo display purposes
        for j in chunk:
            if j == 0:
                row.append("_")
            else:
                row.append(j)
        print(*row)        


def moves(board):
    '''
    Generates all valid successor states from the given board configuration.

    The blank tile can move up, down, left, or right,
    provided the move stays within board boundaries.

    Returns:
        A list of tuples (new_board, direction), 
        where new_board is the resulting configuration 
        and direction indicates the move taken.
    '''
    neighbors = []

    # Locate a blank tile and compute its row and column
    blank = board.index(0)
    row = blank // 3
    col = blank % 3

    # If blank is not in the rightmost column, it can move right
    if col < 2:
        right = blank + 1
        new_list= list(board)
        new_list[blank], new_list[right] = new_list[right], new_list[blank]
        new_board = tuple(new_list)
        neighbors.append((new_board, "Right"))

    # If blank is not in the leftmost column, it can move left
    if col > 0:
        left = blank - 1
        new_list = list(board)
        new_list[blank], new_list[left] = new_list[left], new_list[blank]
        new_board = tuple(new_list)
        neighbors.append((new_board, "Left"))

    # If blank is not in the top column, it can move up
    if row > 0:
        up = blank - 3
        new_list = list(board)
        new_list[blank], new_list[up] = new_list[up], new_list[blank]
        new_board = tuple(new_list)
        neighbors.append((new_board, "Up"))

    # If blank is not in the bottom column, it can move down.
    if row < 2:
        down = blank + 3
        new_list = list(board)
        new_list[blank], new_list[down] = new_list[down], new_list[blank]
        new_board = tuple(new_list)
        neighbors.append((new_board, "Down"))

    return neighbors


def print_path(goal):
    '''
    Reconstructs and prints the solution path from the initial state to the goal.

    The search algorithms (BFS/IDFS) store parent references in each Node.
    Starting from the goal node, this function follows parent pointers
    back to the root, collectsthe states, and then 
    reverses the order to display the path from start to goal.

    Each state is printed witha step number indicating its depth
    in the solution sequence
    '''
    path = []

    # Traverse backwards from the goal to the initial state using parent references
    current = goal
    while current is not None:
        path.append(current.state)
        current = current.parent

    # Reverse the path so it prints from start to goal
    path.reverse()

    # Print each state in order with step number
    for step_number, node_state in enumerate(path):
        print(f"Step {step_number}:")
        print_state(node_state)
        print()


def print_section(title):
    '''
    Formatting to note different sections in output
    '''
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50 + "\n")

def print_expansions(expansion_count, node):
    '''
    Formating for displaying each expansion.

    Prints expansion number, depth, and state for each node.
    '''
    print(f"Expansion #: {expansion_count}")
    print(f"Depth: {node.depth}")
    print_state(node.state)
    print()


In [66]:
def manhattan_distance(state, goal):
    '''
    Calculates the total Manhattan Distance heuristic for the board.

    Manhattan Distance = sum of the row/column distance of each tile
    from its goal position.
    '''
    distance = 0

    for value in range(1, 9):
        current_index = state.index(value)
        goal_index = goal.index(value)

        current_row = current_index // 3
        current_col = current_index % 3

        goal_row = goal_index // 3
        goal_col = goal_index % 3

        distance += abs(current_row - goal_row) + abs(current_col - goal_col)

    return distance

import heapq


def a_star(start_node, goal):
    '''
    A* Search.
    Expands the node with the lowest f(n) = g(n) + h(n).

    g(n) = path cost from the start state
    h(n) = Manhattan Distance heuristic

    Uses a priority queue for the frontier and a reached dictionary
    to track the best cost found so far for each state.
    '''
    print_section("BEGIN SEARCH TREE (First 100 Expansions)")

    # If initial state is already the goal, return immediately
    if start_node.state == goal:
        print_state(start_node.state)
        return start_node

    frontier = []
    counter = 0
    expansion_count = 0

    # Starting node costs
    g_cost = 0
    h_cost = manhattan_distance(start_node.state, goal)
    f_cost = g_cost + h_cost

    # Push starting node into priority queue
    heapq.heappush(frontier, (f_cost, counter, start_node))

    # Track the lowest path cost found so far for each state
    reached = {}
    reached[start_node.state] = 0

    while frontier:
        # Pop node with lowest f(n)
        current_f,_, current = heapq.heappop(frontier)

        expansion_count += 1
        if expansion_count <= 100:
            print_expansions(expansion_count, current)

        # Goal test on expansion
        if current.state == goal:
            print_section("SEARCH COMPLETE")
            print(f"TOTAL EXPANSIONS: {expansion_count}")
            print(f"SOLUTION DEPTH: {current.depth}")
            print_section("SOLUTION PATH")
            return current

        # Expand current node by generating successors
        for child_state, direction in moves(current.state):
            new_g = current.depth + 1

            # Only continue if this is a new state
            # or a cheaper path to an existing state
            if child_state not in reached or new_g < reached[child_state]:
                reached[child_state] = new_g

                child = Node(
                    parent=current,
                    state=child_state,
                    move=direction,
                    depth=new_g
                )

                h_cost = manhattan_distance(child_state, goal)
                f_cost = new_g + h_cost

                counter += 1
                heapq.heappush(frontier, (f_cost, counter, child))

    return None

In [67]:
GOAL = (1, 2, 3, 4, 0, 5, 6, 7, 8)

start_state = random_board_generator()

node = Node(start_state)
result = a_star(node, GOAL)
print_path(result)


BEGIN SEARCH TREE (First 100 Expansions)

Expansion #: 1
Depth: 0
3 6 5
_ 4 1
7 2 8

Expansion #: 2
Depth: 1
3 6 5
4 _ 1
7 2 8

Expansion #: 3
Depth: 2
3 6 5
4 1 _
7 2 8

Expansion #: 4
Depth: 2
3 _ 5
4 6 1
7 2 8

Expansion #: 5
Depth: 2
3 6 5
4 2 1
7 _ 8

Expansion #: 6
Depth: 3
3 6 _
4 1 5
7 2 8

Expansion #: 7
Depth: 3
_ 3 5
4 6 1
7 2 8

Expansion #: 8
Depth: 3
3 6 5
4 2 1
_ 7 8

Expansion #: 9
Depth: 1
_ 6 5
3 4 1
7 2 8

Expansion #: 10
Depth: 1
3 6 5
7 4 1
_ 2 8

Expansion #: 11
Depth: 3
3 6 5
4 1 8
7 2 _

Expansion #: 12
Depth: 3
3 5 _
4 6 1
7 2 8

Expansion #: 13
Depth: 3
3 6 5
4 2 1
7 8 _

Expansion #: 14
Depth: 4
3 _ 6
4 1 5
7 2 8

Expansion #: 15
Depth: 4
4 3 5
_ 6 1
7 2 8

Expansion #: 16
Depth: 4
3 6 5
_ 2 1
4 7 8

Expansion #: 17
Depth: 2
6 _ 5
3 4 1
7 2 8

Expansion #: 18
Depth: 4
3 5 1
4 6 _
7 2 8

Expansion #: 19
Depth: 5
_ 3 6
4 1 5
7 2 8

Expansion #: 20
Depth: 5
3 1 6
4 _ 5
7 2 8

Expansion #: 21
Depth: 5
4 3 5
6 _ 1
7 2 8

Expansion #: 22
Depth: 6
3 1 6
4 2 5
7 _ 8